# 05 — Build ATOMIC SFT datasets from local CSV splits

Metti i file ATOMIC v4 qui:
- `data/raw/atomic/v4_atomic_trn.csv`
- `data/raw/atomic/v4_atomic_dev.csv`
- `data/raw/atomic/v4_atomic_tst.csv`

Output in `data/processed/`:
- `sft_teacher_atomic_train.jsonl`
- `sft_teacher_atomic_val.jsonl`
- `sft_teacher_atomic_test.jsonl`


In [1]:
from pathlib import Path
PROJECT_ROOT = Path('..').resolve()
RAW_ATOMIC_DIR = PROJECT_ROOT / 'data' / 'raw' / 'atomic'
PROC_DIR = PROJECT_ROOT / 'data' / 'processed'
RAW_ATOMIC_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)
RAW_ATOMIC_DIR, PROC_DIR

(WindowsPath('C:/Users/cola0/Desktop/nlp.project-Colangelo-2526/data/raw/atomic'),
 WindowsPath('C:/Users/cola0/Desktop/nlp.project-Colangelo-2526/data/processed'))

In [2]:
train_csv = RAW_ATOMIC_DIR / 'v4_atomic_trn.csv'
dev_csv   = RAW_ATOMIC_DIR / 'v4_atomic_dev.csv'
test_csv  = RAW_ATOMIC_DIR / 'v4_atomic_tst.csv'
train_csv, dev_csv, test_csv

(WindowsPath('C:/Users/cola0/Desktop/nlp.project-Colangelo-2526/data/raw/atomic/v4_atomic_trn.csv'),
 WindowsPath('C:/Users/cola0/Desktop/nlp.project-Colangelo-2526/data/raw/atomic/v4_atomic_dev.csv'),
 WindowsPath('C:/Users/cola0/Desktop/nlp.project-Colangelo-2526/data/raw/atomic/v4_atomic_tst.csv'))

In [3]:
import sys
sys.path.append(str((PROJECT_ROOT / 'src').resolve()))
from data.atomic_local_to_sft import AtomicLocalBuildConfig, build_atomic_local_sft
from utils.jsonl import write_jsonl

In [4]:
cfg = AtomicLocalBuildConfig(
    train_csv=train_csv,
    dev_csv=dev_csv,
    test_csv=test_csv,
    max_rows_train=20_000,
    max_rows_val=5_000,
    max_rows_test=5_000,
    max_completions_per_event_relation=2,
    drop_none=True,
    seed=1234,
)
cfg

AtomicLocalBuildConfig(train_csv=WindowsPath('C:/Users/cola0/Desktop/nlp.project-Colangelo-2526/data/raw/atomic/v4_atomic_trn.csv'), dev_csv=WindowsPath('C:/Users/cola0/Desktop/nlp.project-Colangelo-2526/data/raw/atomic/v4_atomic_dev.csv'), test_csv=WindowsPath('C:/Users/cola0/Desktop/nlp.project-Colangelo-2526/data/raw/atomic/v4_atomic_tst.csv'), system_prompt='You are a helpful assistant specialized in commonsense knowledge completion. Given an event and a relation type, generate ONE plausible completion phrase. Return ONLY the completion phrase.', seed=1234, max_rows_train=20000, max_rows_val=5000, max_rows_test=5000, max_completions_per_event_relation=2, drop_none=True, keep_relations=None)

In [5]:
splits = build_atomic_local_sft(cfg)
for k,v in splits.items():
    print(k, len(v))
splits['train'][0]

train 55736
val 13879
test 13931


{'id': 'atomic_train_0',
 'split': 'train',
 'source': 'atomic_local',
 'event': "PersonX kills PersonX's father",
 'relation': 'oEffect',
 'messages': [{'role': 'system',
   'content': 'You are a helpful assistant specialized in commonsense knowledge completion. Given an event and a relation type, generate ONE plausible completion phrase. Return ONLY the completion phrase.'},
  {'role': 'user',
   'content': "Event: PersonX kills PersonX's father\nRelation: oEffect\nCompletion:"}],
 'assistant': 'bleeds'}

In [6]:
out_train = PROC_DIR / 'sft_teacher_atomic_train.jsonl'
out_val   = PROC_DIR / 'sft_teacher_atomic_val.jsonl'
out_test  = PROC_DIR / 'sft_teacher_atomic_test.jsonl'
write_jsonl(splits['train'], out_train)
write_jsonl(splits['val'], out_val)
write_jsonl(splits['test'], out_test)
print('Saved:', out_train)
print('Saved:', out_val)
print('Saved:', out_test)

Saved: C:\Users\cola0\Desktop\nlp.project-Colangelo-2526\data\processed\sft_teacher_atomic_train.jsonl
Saved: C:\Users\cola0\Desktop\nlp.project-Colangelo-2526\data\processed\sft_teacher_atomic_val.jsonl
Saved: C:\Users\cola0\Desktop\nlp.project-Colangelo-2526\data\processed\sft_teacher_atomic_test.jsonl
